# NL2P-1 Results Presentation

Presentation notebook for comparing current `nl2p_1`, `nl2p_1_ablation`, and `nl2p_1_coref` results.

Focus:
- model performance across available runs
- prompt / strategy performance
- action extraction vs object argument extraction
- diagnostics buckets for remaining errors


In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "results").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing results/ and src/.")


ROOT = find_repo_root()
RESULTS = ROOT / "results"

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

ROOT


## Load Current Result Files

The notebook reads result directories directly from `results/`.

`strategy` is inferred from the directory name, not only from the CSV columns, because coref runs may keep `solver == nl2p_1` inside the evaluator output.


In [ ]:
TARGET_STRATEGIES = {
    "nl2p_1": {
        "prompt": "baseline",
        "input": "original",
        "strategy": "baseline prompt",
        "label": "NL2P-1",
    },
    "nl2p_1_ablation": {
        "prompt": "ablation",
        "input": "original",
        "strategy": "ablation prompt",
        "label": "NL2P-1 ablation",
    },
    "nl2p_1_coref": {
        "prompt": "baseline",
        "input": "coref-resolved",
        "strategy": "coref input",
        "label": "NL2P-1 + coref",
    },
    "nl2p_1_ablation_coref": {
        "prompt": "ablation",
        "input": "coref-resolved",
        "strategy": "ablation + coref",
        "label": "NL2P-1 ablation + coref",
    },
}

METRICS = [
    "Precision",
    "Recall",
    "F1",
    "Object Precision",
    "Object Recall",
    "Object F1",
]


def read_eval_result(path):
    df = pd.read_csv(path)
    run_dir = path.parent
    strategy_key = run_dir.parent.name
    model = run_dir.name
    meta = TARGET_STRATEGIES[strategy_key]
    df = df.copy()
    df["result_path"] = str(path.relative_to(ROOT))
    df["strategy_key"] = strategy_key
    df["strategy"] = meta["strategy"]
    df["prompt"] = meta["prompt"]
    df["input_mode"] = meta["input"]
    df["strategy_label"] = meta["label"]
    df["model"] = model
    df["run_label"] = meta["label"] + " / " + model
    return df


eval_paths = []
for strategy_key in TARGET_STRATEGIES:
    strategy_dir = RESULTS / strategy_key
    if strategy_dir.exists():
        eval_paths.extend(sorted(strategy_dir.glob("*/evaluation_result.csv")))

eval_df = pd.concat([read_eval_result(path) for path in eval_paths], ignore_index=True)
for metric in METRICS:
    eval_df[metric] = pd.to_numeric(eval_df[metric], errors="coerce")

coverage = (
    eval_df.groupby(["strategy_label", "model"])["dataset"]
    .agg(lambda s: ", ".join(sorted(s.unique())))
    .reset_index(name="datasets")
    .sort_values(["strategy_label", "model"])
)

print(f"Loaded {len(eval_paths)} evaluation files and {len(eval_df)} dataset rows.")
display(coverage)


## Overall Model Performance

Mean over available datasets. Use this as the first high-level slide: action F1 shows event extraction quality, object F1 shows argument quality.


In [ ]:
summary = (
    eval_df.groupby(["run_label", "strategy_label", "strategy", "prompt", "input_mode", "model"], as_index=False)[METRICS]
    .mean()
    .sort_values(["F1", "Object F1"], ascending=False)
)

display(
    summary[["run_label", "Precision", "Recall", "F1", "Object Precision", "Object Recall", "Object F1"]]
    .style.format({metric: "{:.3f}" for metric in METRICS})
)


In [ ]:
plot_df = summary.melt(
    id_vars=["run_label", "strategy_label", "strategy", "prompt", "input_mode", "model"],
    value_vars=["F1", "Object F1"],
    var_name="metric",
    value_name="score",
)
order = summary.sort_values("F1", ascending=False)["run_label"]

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot_df, y="run_label", x="score", hue="metric", order=order, ax=ax)
ax.set_xlim(0, 1)
ax.set_xlabel("Mean score across available datasets")
ax.set_ylabel("")
ax.set_title("Overall performance: action F1 vs object F1")
ax.legend(title="")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=9)
plt.tight_layout()
plt.show()


## Action/Object Trade-off

This scatter plot is useful for explaining whether a run improves action extraction only, argument extraction only, or both.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
sns.scatterplot(
    data=summary,
    x="F1",
    y="Object F1",
    hue="strategy_label",
    style="model",
    s=160,
    ax=ax,
)
for _, row in summary.iterrows():
    ax.annotate(row["model"], (row["F1"], row["Object F1"]), xytext=(6, 4), textcoords="offset points", fontsize=9)
ax.set_xlim(max(0, summary["F1"].min() - 0.04), min(1, summary["F1"].max() + 0.04))
ax.set_ylim(max(0, summary["Object F1"].min() - 0.04), min(1, summary["Object F1"].max() + 0.04))
ax.set_xlabel("Action F1")
ax.set_ylabel("Object F1")
ax.set_title("Action extraction vs argument extraction")
ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Dataset-level Comparison

The model ranking is not uniform across `cooking`, `wikihow`, and `win2k`; this plot is intended for a dataset-specific comparison slide.


In [ ]:
dataset_plot = eval_df.melt(
    id_vars=["dataset", "run_label", "strategy_label", "model"],
    value_vars=["F1", "Object F1"],
    var_name="metric",
    value_name="score",
)

g = sns.catplot(
    data=dataset_plot,
    kind="bar",
    x="dataset",
    y="score",
    hue="run_label",
    col="metric",
    height=5,
    aspect=1.25,
    sharey=True,
)
g.set_axis_labels("", "Score")
g.set_titles("{col_name}")
for ax in g.axes.flat:
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=0)
g.legend.set_title("")
plt.subplots_adjust(top=0.82)
g.fig.suptitle("Performance by dataset")
plt.show()


## Prompt and Strategy Deltas

Delta plots compare each strategy with the matching baseline `nl2p_1` run for the same model and dataset.

Positive values mean the strategy improved over the baseline.


In [ ]:
baseline = eval_df[eval_df["strategy_key"] == "nl2p_1"][
    ["dataset", "model", *METRICS]
].rename(columns={metric: f"{metric}_baseline" for metric in METRICS})

delta_df = eval_df[eval_df["strategy_key"] != "nl2p_1"].merge(
    baseline,
    on=["dataset", "model"],
    how="inner",
)
for metric in METRICS:
    delta_df[f"delta_{metric}"] = delta_df[metric] - delta_df[f"{metric}_baseline"]

delta_metrics = ["delta_Precision", "delta_Recall", "delta_F1", "delta_Object F1"]
display_cols = ["strategy_label", "model", "dataset", *delta_metrics]
display(
    delta_df[display_cols]
    .sort_values(["strategy_label", "model", "dataset"])
    .style.format({metric: "{:+.3f}" for metric in delta_metrics})
)


In [ ]:
if delta_df.empty:
    print("No matching strategy-vs-baseline pairs are available yet.")
else:
    delta_plot = delta_df.melt(
        id_vars=["strategy_label", "model", "dataset", "run_label"],
        value_vars=["delta_F1", "delta_Object F1"],
        var_name="metric",
        value_name="delta",
    )
    delta_plot["metric"] = delta_plot["metric"].str.replace("delta_", "", regex=False)
    delta_plot["comparison"] = delta_plot["strategy_label"] + " / " + delta_plot["model"]

    g = sns.catplot(
        data=delta_plot,
        kind="bar",
        x="dataset",
        y="delta",
        hue="metric",
        row="comparison",
        height=2.4,
        aspect=3.0,
        sharey=True,
    )
    for ax in g.axes.flat:
        ax.axhline(0, color="black", linewidth=1)
        ax.set_xlabel("")
        ax.set_ylabel("Delta")
    g.legend.set_title("")
    g.set_titles("{row_name}")
    plt.subplots_adjust(top=0.93)
    g.fig.suptitle("Strategy delta vs matching NL2P-1 baseline")
    plt.show()


## Precision/Recall Movement

Use this for explaining the common trade-off: ablation may raise recall while lowering precision; coref may change argument recovery differently from action detection.


In [ ]:
pair_df = eval_df.copy()
pair_df["metric_pair"] = "Action"
action_pr = pair_df[["dataset", "run_label", "strategy_label", "model", "Precision", "Recall"]].rename(
    columns={"Precision": "precision", "Recall": "recall"}
)
action_pr["metric_pair"] = "Action"
object_pr = pair_df[["dataset", "run_label", "strategy_label", "model", "Object Precision", "Object Recall"]].rename(
    columns={"Object Precision": "precision", "Object Recall": "recall"}
)
object_pr["metric_pair"] = "Object"
pr_df = pd.concat([action_pr, object_pr], ignore_index=True)

g = sns.relplot(
    data=pr_df,
    x="recall",
    y="precision",
    hue="run_label",
    style="strategy_label",
    col="metric_pair",
    row="dataset",
    s=120,
    height=3.2,
    aspect=1.25,
)
for ax in g.axes.flat:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
g.set_axis_labels("Recall", "Precision")
g.set_titles("{row_name} / {col_name}")
g.legend.set_title("")
plt.show()


## Diagnostics Overview

Diagnostics are available only for runs where `evaluation.py --diagnostics` has been run. The plots below compare mismatch buckets and strong dataset-side issue signals.


In [ ]:
def read_diagnostics(path):
    df = pd.read_csv(path)
    run_dir = path.parent
    strategy_key = run_dir.parent.name
    model = run_dir.name
    meta = TARGET_STRATEGIES.get(strategy_key, {})
    df = df.copy()
    df["strategy_key"] = strategy_key
    df["strategy_label"] = meta.get("label", strategy_key)
    df["strategy"] = meta.get("strategy", strategy_key)
    df["model"] = model
    df["run_label"] = df["strategy_label"] + " / " + model
    df["result_path"] = str(path.relative_to(ROOT))
    return df


diag_paths = []
for strategy_key in TARGET_STRATEGIES:
    strategy_dir = RESULTS / strategy_key
    if strategy_dir.exists():
        diag_paths.extend(sorted(strategy_dir.glob("*/evaluation_mismatch_diagnostics.csv")))

if diag_paths:
    diag_df = pd.concat([read_diagnostics(path) for path in diag_paths], ignore_index=True)
    print(f"Loaded {len(diag_paths)} diagnostics files and {len(diag_df)} diagnostic rows.")
    display(
        diag_df.groupby(["run_label", "dataset"])["mismatch_type"]
        .size()
        .reset_index(name="diagnostic_rows")
        .sort_values(["run_label", "dataset"])
    )
else:
    diag_df = pd.DataFrame()
    print("No diagnostics CSV files found for the target strategies.")


In [ ]:
if not diag_df.empty:
    mismatch_counts = (
        diag_df.groupby(["run_label", "mismatch_type"])
        .size()
        .reset_index(name="count")
    )
    mismatch_counts["share"] = mismatch_counts["count"] / mismatch_counts.groupby("run_label")["count"].transform("sum")

    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
    sns.barplot(data=mismatch_counts, y="run_label", x="count", hue="mismatch_type", ax=axes[0])
    axes[0].set_title("Diagnostic rows")
    axes[0].set_xlabel("Count")
    axes[0].set_ylabel("")
    sns.barplot(data=mismatch_counts, y="run_label", x="share", hue="mismatch_type", ax=axes[1])
    axes[1].set_title("Diagnostic row share")
    axes[1].set_xlabel("Share")
    axes[1].set_ylabel("")
    axes[0].legend_.remove()
    axes[1].legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
def split_issue_components(value):
    if pd.isna(value) or not str(value).strip():
        return []
    parts = [part.strip() for part in str(value).split("|") if part.strip()]
    detailed = [part for part in parts if ":" in part]
    return detailed or parts


if not diag_df.empty and "strong_dataset_issue" in diag_df.columns:
    strong = diag_df[diag_df["strong_dataset_issue"].fillna("").astype(str).str.len() > 0].copy()
    if strong.empty:
        print("No strong dataset-side diagnostic rows in the loaded diagnostics.")
    else:
        strong["issue_component"] = strong["strong_dataset_issue"].apply(split_issue_components)
        strong = strong.explode("issue_component")
        strong_counts = (
            strong.groupby(["run_label", "issue_component"])
            .size()
            .reset_index(name="count")
            .sort_values(["run_label", "count"], ascending=[True, False])
        )
        display(strong_counts)

        fig, ax = plt.subplots(figsize=(12, 6))
        sns.barplot(data=strong_counts, y="run_label", x="count", hue="issue_component", ax=ax)
        ax.set_title("Strong dataset-side diagnostic signals")
        ax.set_xlabel("Rows")
        ax.set_ylabel("")
        ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()


## Presentation Takeaways

Use the tables below to draft slide bullets from current data.


In [ ]:
best_action = summary.sort_values("F1", ascending=False).head(5)[["run_label", "F1", "Precision", "Recall"]]
best_object = summary.sort_values("Object F1", ascending=False).head(5)[["run_label", "Object F1", "Object Precision", "Object Recall"]]

print("Top runs by mean action F1")
display(best_action.style.format({"F1": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}"}))

print("Top runs by mean object F1")
display(best_object.style.format({"Object F1": "{:.3f}", "Object Precision": "{:.3f}", "Object Recall": "{:.3f}"}))

if not delta_df.empty:
    delta_summary = (
        delta_df.groupby(["strategy_label", "model"], as_index=False)[["delta_F1", "delta_Object F1"]]
        .mean()
        .sort_values(["delta_F1", "delta_Object F1"], ascending=False)
    )
    print("Mean delta vs matching baseline")
    display(delta_summary.style.format({"delta_F1": "{:+.3f}", "delta_Object F1": "{:+.3f}"}))
